<a href="https://colab.research.google.com/github/fonglieu/text-analytics-assignment5/blob/main/Assignment_5_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 5 — Option B: Job Fit Analyzer
## BSAN 6200: Text Mining & Social Media Analytics — Spring 2026

**Student Name:** Fong Lieu
**Option:** B — Job Fit Analyzer  
**API Path:** [Paid / Free]  

---

### Table of Contents
1. [Setup and Imports](#1-setup)
2. [Load Job Descriptions and Resume](#2-loading)
3. [Text Chunking](#3-chunking)
4. [Embedding and Vector Store](#4-embedding)
5. [Analysis Prompts and Chain](#5-analysis)
6. [Zero-shot vs. Few-shot Comparison](#6-comparison)
7. [Evaluation](#7-evaluation)

> **Reminder:** The Streamlit app is a separate file (`streamlit_app.py`). This notebook builds and tests the analysis pipeline.  
> See the Option B Implementation Guide for detailed step requirements.

---
<a id="1-setup"></a>
## 1. Setup and Imports

Install required packages and load your API key from a `.env` file.  
**Do NOT hardcode API keys in this notebook.**

Suggested packages: `langchain`, `langchain-openai` or `langchain-community`, `chromadb` or `faiss-cpu`, `pypdf`, `python-dotenv`, `pandas`, `sentence-transformers` (free path)

In [16]:
# ── Install packages (uncomment as needed) ──
!pip install openai pypdf pandas chromadb sentence-transformers -q

from google.colab import drive
drive.mount('/content/drive')

# ── Your imports below ──
import os
import re
import pandas as pd
from collections import Counter
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from openai import OpenAI
from google.colab import userdata

# ── API key ──
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# ── Paths ──
BASE_DIR = "/content/drive/MyDrive/data"

print("Imports successful.")
print(f"Files: {os.listdir(BASE_DIR)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Imports successful.
Files: ['Fong_Lieu_Resume.pdf', 'jd_08_capgemini_junior_ba.txt', 'jd_03_spotify_data_analyst.txt', 'jd_10_goldmansachs_data_office_analyst.txt', 'jd_09_bayview_data_ops_analyst.txt', 'jd_02_att_strategy_analyst.txt', 'jd_04_delta_data_analyst.txt', 'jd_06_transformlabs_business_analyst.txt', 'jd_metadata.csv', 'jd_05_northrop_grumman_data_analyst.txt', 'jd_01_amend_sales_analyst.txt', 'jd_07_qualifiedhealth_product_intern.txt', 'chroma_db']


---
<a id="2-loading"></a>
## 2. Load Job Descriptions and Resume

**Required:**
- 10+ JD files in `data/job_descriptions/` (each as a separate .txt or .pdf)
- Your resume in `data/resume/`
- A metadata file `data/jd_metadata.csv` with columns: filename, company, title, source_url, date_collected

Print: number of JDs loaded, number of resume docs, and preview content from each.

In [17]:
# ── Load JD metadata ──
metadata_df = pd.read_csv(os.path.join(BASE_DIR, "jd_metadata.csv"))
print(f"JDs loaded: {len(metadata_df)}\n")
display(metadata_df)

JDs loaded: 10



,filename,company,title,source_url,date_collected
0,jd_01_amend_sales_analyst.txt,AMEND Consulting,Sales Analyst – Co-op/Internship,https://job-boards.greenhouse.io/amendconsulti...,2026-05-09
1,jd_02_att_strategy_analyst.txt,AT&T,Strategy Analyst – Wireline Transformation & S...,https://www.att.jobs/job/-/strategy-analyst/11...,2026-05-09
2,jd_03_spotify_data_analyst.txt,Spotify,Data Analyst II – Performance Optimization Squad,https://jobs.lever.co/spotify/d95e1989-9a1a-48...,2026-05-09
3,jd_04_delta_data_analyst.txt,Delta Air Lines,Senior Analyst – Customer Data Solutions,https://delta.avature.net/en_US/careers/JobDet...,2026-05-09
4,jd_05_northrop_grumman_data_analyst.txt,Northrop Grumman,Data Insight Analyst,https://jobs.northropgrumman.com/careers/job/1...,2026-05-09
5,jd_06_transformlabs_business_analyst.txt,Transform Labs,Entry-Level Business Analyst,https://awhnet.bamboohr.com/careers/76,2026-05-09
6,jd_07_qualifiedhealth_product_intern.txt,Qualified Health,Product Intern,https://lmu.joinhandshake.com/jobs/10994147,2026-05-09
7,jd_08_capgemini_junior_ba.txt,Capgemini America Inc.,Junior Business Analyst,https://lmu.joinhandshake.com/job-search/10982553,2026-05-09
8,jd_09_bayview_data_ops_analyst.txt,Bayview Asset Management,Data Operations Analyst,https://lmu.joinhandshake.com/job-search/10974135,2026-05-09
9,jd_10_goldmansachs_data_office_analyst.txt,Goldman Sachs,Asset & Wealth Management Data Office Analyst,https://www.linkedin.com/jobs/view/4375782060/,2026-05-09


In [18]:
# ── Load JD documents ──
def load_text_file(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return f.read()

jd_docs = {}
for _, row in metadata_df.iterrows():
    filepath = os.path.join(BASE_DIR, row["filename"])
    jd_docs[row["filename"]] = {
        "text":       load_text_file(filepath),
        "company":    row["company"],
        "title":      row["title"],
        "source_url": row["source_url"]
    }

print(f"JD documents loaded: {len(jd_docs)}")

JD documents loaded: 10


In [19]:
# ── Load Resume ──
def load_pdf(filepath):
    reader = PdfReader(filepath)
    return "\n".join([page.extract_text() for page in reader.pages if page.extract_text()])

resume_text = load_pdf(os.path.join(BASE_DIR, "Fong_Lieu_Resume.pdf"))
print(f"Resume loaded. Length: {len(resume_text)} chars")

Resume loaded. Length: 3438 chars


In [20]:
# ── Preview sample content ──
sample_key = list(jd_docs.keys())[0]
print(f"=== Sample JD: {sample_key} ===")
print(jd_docs[sample_key]["text"][:500])
print("\n=== Resume (first 500 chars) ===")
print(resume_text[:500])

=== Sample JD: jd_01_amend_sales_analyst.txt ===
Company: AMEND Consulting
Title: Sales Analyst – Co-op/Internship
Location: Cincinnati, OH
Source: https://job-boards.greenhouse.io/amendconsulting/jobs/4005828009
Date Collected: 2026-05-09

--- JOB DESCRIPTION ---

About AMEND:
AMEND is a management consulting firm based in Cincinnati, OH with areas of focus in operations, analytics, and technology. We are focused on strengthening the people, processes, and systems in organizations to generate a holistic transformation. Our three-tiered approa

=== Resume (first 500 chars) ===
Fong  Lieu
 
Los  Angeles,  CA   |   (720)  921-4111   |   fonglieu8@gmail.com   
TECHNICAL  SKILLS  Programming  &  Statistical  Languages:  SQL,  Python,  R  Libraries  &  Frameworks:  pandas,  NumPy,  matplotlib,  scikit-learn,  dplyr,  tidyr  Data  Analysis  &  Visualization:  Tableau,  Power  BI,  Excel  Data  Management:  Data  Extraction  &  Transformation,  Databases,  Salesforce  EDUCATION  Loyola  Marym

---
<a id="3-chunking"></a>
## 3. Text Chunking

Split your documents into chunks.  
**Required:** Try at least 2 chunking strategies, compare them quantitatively, and justify your final choice.

**Hint:** JDs often have natural sections (Requirements, Responsibilities, Qualifications). Consider whether your splitter respects these boundaries.

In [21]:
# ── Strategy 1 ──
def chunk_text(text, chunk_size=600, overlap=75):
    """Strategy 1: fixed-size, may split mid-sentence."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk.strip())
        start += chunk_size - overlap
    return [c for c in chunks if len(c) > 20]

# Run Strategy 1 on all documents
chunks_1 = []
for filename, doc in jd_docs.items():
    chunks_1.extend(chunk_text(doc["text"], chunk_size=600, overlap=75))
chunks_1.extend(chunk_text(resume_text, chunk_size=600, overlap=75))

print("Strategy 1: Fixed-Size Chunking")
print(f"  Total chunks:      {len(chunks_1)}")
print(f"  Avg length (chars): {round(sum(len(c) for c in chunks_1) / len(chunks_1))}")
print(f"  Min chunk:          {min(len(c) for c in chunks_1)}")
print(f"  Max chunk:          {max(len(c) for c in chunks_1)}")
print(f"\nSample chunk:")
print(chunks_1[3])

Strategy 1: Fixed-Size Chunking
  Total chunks:      74
  Avg length (chars): 544
  Min chunk:          33
  Max chunk:          600

Sample chunk:
aintenance of sales activity data in Salesforce and other key systems
- Prepare weekly sales reports and analyze data through Excel manipulation to track key performance indicators (KPIs)
- Generate and elevate insights on ways to improve sales support processes, systems, and data
- Support the creation and organization of sales materials, inclusive of presentations, one-pagers, and more
- Own and execute elements of event planning and delivery within the sales function
- Take on additional responsibilities as needed to support the broader sales and business development efforts

Qualifications


In [22]:
# ── Strategy 2 ──
def chunk_text_by_sentences(text, chunk_size=600, overlap=75):
    """Strategy 2: respects sentence boundaries."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks = []
    current_chunk = ""
    for sentence in sentences:
        if len(current_chunk) + len(sentence) > chunk_size and current_chunk:
            chunks.append(current_chunk.strip())
            words = current_chunk.split()
            overlap_text = " ".join(words[-10:]) if len(words) > 10 else current_chunk
            current_chunk = overlap_text + " " + sentence
        else:
            current_chunk += (" " if current_chunk else "") + sentence
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    return chunks

# Run Strategy 2 on all documents, store with metadata
all_chunks = []
for filename, doc in jd_docs.items():
    doc_chunks = chunk_text_by_sentences(doc["text"], chunk_size=600, overlap=75)
    for chunk in doc_chunks:
        all_chunks.append({
            "text":    chunk,
            "source":  filename,
            "company": doc["company"]
        })

resume_chunks = chunk_text_by_sentences(resume_text, chunk_size=600, overlap=75)
for chunk in resume_chunks:
    all_chunks.append({
        "text":    chunk,
        "source":  "Fong_Lieu_Resume.pdf",
        "company": "resume"
    })

chunks_2 = [c["text"] for c in all_chunks]

print("Strategy 2: Sentence-Aware Chunking")
print(f"  Total chunks:       {len(chunks_2)}")
print(f"  Avg length (chars): {round(sum(len(c) for c in chunks_2) / len(chunks_2))}")
print(f"  Min chunk:          {min(len(c) for c in chunks_2)}")
print(f"  Max chunk:          {max(len(c) for c in chunks_2)}")
print(f"\nChunks per source:")
for source, count in Counter(c["source"] for c in all_chunks).items():
    print(f"  {source}: {count}")
print(f"\nSample chunk:")
print(chunks_2[3])

Strategy 2: Sentence-Aware Chunking
  Total chunks:       44
  Avg length (chars): 866
  Min chunk:          220
  Max chunk:          2750

Chunks per source:
  jd_01_amend_sales_analyst.txt: 4
  jd_02_att_strategy_analyst.txt: 3
  jd_03_spotify_data_analyst.txt: 4
  jd_04_delta_data_analyst.txt: 3
  jd_05_northrop_grumman_data_analyst.txt: 4
  jd_06_transformlabs_business_analyst.txt: 3
  jd_07_qualifiedhealth_product_intern.txt: 3
  jd_08_capgemini_junior_ba.txt: 4
  jd_09_bayview_data_ops_analyst.txt: 4
  jd_10_goldmansachs_data_office_analyst.txt: 6
  Fong_Lieu_Resume.pdf: 6

Sample chunk:
program with curriculum to build a wholistic understanding of business. Responsibilities:
- Develop a strong understanding of AMEND's service offerings and approach to client engagement
- Research and identify potential sales leads and market trends to support an active and organized pipeline for the sales team, using tools such as ZoomInfo and Hunter
- Ensure accurate entry and ongoing maintena

In [23]:
# ── Compare strategies ──
comparison = pd.DataFrame({
    "Strategy": [
        "Strategy 1: Fixed-Size (600/75)",
        "Strategy 2: Sentence-Aware (600/75)"
    ],
    "Total Chunks": [len(chunks_1), len(chunks_2)],
    "Avg Length (chars)": [
        round(sum(len(c) for c in chunks_1) / len(chunks_1)),
        round(sum(len(c) for c in chunks_2) / len(chunks_2))
    ],
    "Min Chunk": [
        min(len(c) for c in chunks_1),
        min(len(c) for c in chunks_2)
    ],
    "Max Chunk": [
        max(len(c) for c in chunks_1),
        max(len(c) for c in chunks_2)
    ]
})
display(comparison)

print("\n=== Strategy 1 sample chunk ===")
print(chunks_1[3])
print("\n=== Strategy 2 sample chunk ===")
print(chunks_2[3])

,Strategy,Total Chunks,Avg Length (chars),Min Chunk,Max Chunk
0,Strategy 1: Fixed-Size (600/75),74,544,33,600
1,Strategy 2: Sentence-Aware (600/75),44,866,220,2750



=== Strategy 1 sample chunk ===
aintenance of sales activity data in Salesforce and other key systems
- Prepare weekly sales reports and analyze data through Excel manipulation to track key performance indicators (KPIs)
- Generate and elevate insights on ways to improve sales support processes, systems, and data
- Support the creation and organization of sales materials, inclusive of presentations, one-pagers, and more
- Own and execute elements of event planning and delivery within the sales function
- Take on additional responsibilities as needed to support the broader sales and business development efforts

Qualifications

=== Strategy 2 sample chunk ===
program with curriculum to build a wholistic understanding of business. Responsibilities:
- Develop a strong understanding of AMEND's service offerings and approach to client engagement
- Research and identify potential sales leads and market trends to support an active and organized pipeline for the sales team, using tools such as

### Chunking Decision

**Which strategy did you choose?**
Strategy 2 — Sentence-Aware Chunking (600/75)

**Why?**
The comparison table shows Strategy 1 produced 74 chunks with an average length
of 544 chars and a minimum chunk of just 33 chars, meaning it frequently created
fragments — pieces of sentences or lone section headers stripped of their
surrounding requirements. The sample chunk from Strategy 1 illustrates this
directly: it begins mid-sentence with "aintenance of sales activity data in
Salesforce..." — a clear mid-word cut that breaks the semantic meaning of the
requirement. Strategy 2 produced 44 chunks with an average of 866 chars and a
minimum of 220 chars, keeping full requirement sections intact. The sample chunk
from Strategy 2 captures the complete Responsibilities and Qualifications sections
of the AMEND JD as a single coherent unit, which means a similarity search returns
a meaningful, complete section rather than a fragment. For a job fit analyzer where
retrieval quality directly determines analysis quality, Strategy 2's
section-preserving behavior is the clear choice.

**Final settings:** chunk_size=600, overlap=75

---
<a id="4-embedding"></a>
## 4. Embedding and Vector Store

Embed your chunks and store them in a vector database (ChromaDB or FAISS).

**Paid path:** OpenAI `text-embedding-3-small`  
**Free path:** `sentence-transformers/all-MiniLM-L6-v2`

After creating the store, run a test similarity search to verify it works.

In [24]:
# ── Create embeddings and vector store ──
import chromadb

chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="job_fit_analyzer",
    metadata={"hnsw:space": "cosine"}
)

if collection.count() == 0:
    collection.add(
        documents=[c["text"]    for c in all_chunks],
        metadatas=[{"source": c["source"], "company": c["company"]} for c in all_chunks],
        ids=[f"chunk_{i}"       for i in range(len(all_chunks))]
    )

print(f"Vector store created with {collection.count()} chunks.")

Vector store created with 44 chunks.


In [25]:
# ── Verify: run a test similarity search ──
def search(query, k=3):
    """Search the vector store and return top-k results with sources and distances."""
    results = collection.query(
        query_texts=[query],
        n_results=k
    )
    docs      = results["documents"][0]
    sources   = [m["source"]  for m in results["metadatas"][0]]
    companies = [m["company"] for m in results["metadatas"][0]]
    distances = results["distances"][0]
    return list(zip(docs, sources, companies, distances))

test_query = "SQL Python data analysis dashboard experience required"
print(f'Query: "{test_query}"\n')

for i, (text, source, company, dist) in enumerate(search(test_query, k=4)):
    print(f"Result {i+1} [source: {company}] (distance: {dist:.3f}):")
    print(f"  {text[:200]}...")
    print()

Query: "SQL Python data analysis dashboard experience required"

Result 1 [source: Northrop Grumman] (distance: 0.326):
  to communicate complex findings in a simple and actionable way. Responsibilities:
- Develop and maintain Tableau dashboards and data visualizations to support operational and strategic decisions
- Use...

Result 2 [source: Delta Air Lines] (distance: 0.391):
  layer and help build scalable solutions that drive business decisions. Responsibilities:
- Analyze large datasets related to customer behavior, scheduling, performance, and operational metrics
- Devel...

Result 3 [source: Spotify] (distance: 0.472):
  and metrics that make performance inefficiencies visible across the platform. Responsibilities:
- Design and own datasets, pipelines, and metrics that expose performance inefficiencies across the plat...

Result 4 [source: Northrop Grumman] (distance: 0.507):
  and logistics and modernization for our customers around the globe. About the Role:
The Data Insight A

---
<a id="5-analysis"></a>
## 5. Analysis Prompts and Chain

Build 3 analysis types, each with its own prompt:

1. **Skill Gap Report:** Compare resume skills vs. JD requirements. Output matching skills, missing skills, and recommended actions.
2. **Keyword Alignment:** Extract key terms from a JD, check which appear in the resume, report a match rate.
3. **Fit Summary:** 3-4 sentence narrative assessment citing evidence from both documents.

You also need to wire up the LLM and a way to pass a specific JD + resume into each prompt.

**Required:** Document at least 3 prompt iterations total (across any analysis type) with rationale.

**Reminder:** Prompt design must be your own work (Tier 2 — AI prohibited for this step).

In [26]:
# ── Initialize LLM ──
def call_llm(prompt, model="gpt-4o-mini", temperature=0):
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [27]:
# Helper Function
def get_jd_text(filename):
    return jd_docs[filename]["text"]

target_jd = get_jd_text("jd_10_goldmansachs_data_office_analyst.txt")
print("Helper ready.")

Helper ready.


In [15]:
# ── Analysis 1: Skill Gap Report ──
def skill_gap_report(jd_text, resume_text):
    prompt = f"""
You are a career coach analyzing a candidate's resume against a job description.

JOB DESCRIPTION:
{jd_text}

CANDIDATE RESUME:
{resume_text}

Produce a structured Skill Gap Report with exactly three sections:

MATCHING SKILLS:
- List each skill or experience the candidate has that directly matches a JD requirement.
- Be specific, citing evidence from the resume (tool name, project, or role).

MISSING SKILLS:
- List each requirement from the JD that is absent or insufficiently demonstrated in the resume.
- Be specific about what is missing.

RECOMMENDED ACTIONS:
- For each missing skill, provide one concrete, actionable step the candidate can take
  (e.g., specific course, project idea, certification, or tool to learn).

Use bullet points. Be concise and factual. Do not invent skills not present in either document.
"""
    return call_llm(prompt)

result = skill_gap_report(target_jd, resume_text)
print("=== SKILL GAP REPORT: Goldman Sachs ===\n")
print(result)

=== SKILL GAP REPORT: Goldman Sachs ===

### SKILL GAP REPORT

#### MATCHING SKILLS:
- **Data Analysis Experience**: The candidate has experience in data extraction and transformation as a Pricing Analyst at Onboard Logistics, which aligns with the requirement for data analysis and operations experience.
- **Technical Skills**: Proficiency in SQL and Python is demonstrated in the resume, matching the job's requirement for these programming languages.
- **Data Visualization**: The candidate has developed KPI dashboards in Tableau, which supports the job's need for preparing dashboards and reports for internal stakeholders.
- **Attention to Detail**: The candidate's work as a Pricing Analyst involved improving win rates and generating profit through data-driven strategies, indicating strong analytical skills and attention to detail.
- **Project Support**: Experience in delivering lessons and improving learning outcomes through data analysis shows the candidate's ability to support projec

In [28]:
# ── Analysis 2: Keyword Alignment ──
def keyword_alignment(jd_text, resume_text):
    prompt = f"""
You are a resume keyword analyst.

JOB DESCRIPTION:
{jd_text}

CANDIDATE RESUME:
{resume_text}

Step 1 — Extract the 15 most important keywords and phrases from the job description
(focus on required skills, tools, technologies, and domain terms).

Step 2 — For each keyword, determine if it appears or is semantically equivalent in the resume.
Mark each as: MATCH, PARTIAL MATCH, or MISSING.

Step 3 — Calculate: match rate = (MATCH + 0.5 * PARTIAL MATCH) / 15

Output format:

KEYWORD TABLE:
| Keyword | Status | Evidence from Resume |
|---------|--------|----------------------|
...

MATCH RATE: X%

SUMMARY: One sentence interpreting the match rate and what it means for this application.
"""
    return call_llm(prompt)

result = keyword_alignment(target_jd, resume_text)
print("=== KEYWORD ALIGNMENT: Goldman Sachs ===\n")
print(result)

=== KEYWORD ALIGNMENT: Goldman Sachs ===

### KEYWORD TABLE:
| Keyword                                      | Status        | Evidence from Resume                                                                 |
|----------------------------------------------|---------------|--------------------------------------------------------------------------------------|
| Data governance                              | MISSING       | No mention of data governance in the resume.                                        |
| Data quality                                 | PARTIAL MATCH | Mention of data-driven identification of performance gaps in the Professional Learning Facilitator role. |
| KPIs                                         | MATCH         | Developed KPI dashboards in Tableau for a team of 5.                                |
| Data validation                               | MISSING       | No specific mention of data validation in the resume.                               |
| Data re

In [29]:
# ── Analysis 3: Fit Summary ──
def fit_summary_v1(jd_text, resume_text):
    prompt = f"""
You are a hiring consultant. Read this job description and resume
and write a short summary of how well the candidate fits the role.

JOB DESCRIPTION:
{jd_text}

CANDIDATE RESUME:
{resume_text}
"""
    return call_llm(prompt)

result_v1 = fit_summary_v1(target_jd, resume_text)
print("=== FIT SUMMARY: Goldman Sachs ===\n")
print(result_v1)

=== FIT SUMMARY: Goldman Sachs ===

**Candidate Fit Summary:**

Fong Lieu presents a strong fit for the Analyst position within Goldman Sachs' Asset & Wealth Management, External Investing Group, Data Office. With a Master's degree in Business Analytics and a Bachelor's in Economics, Fong has a solid educational foundation relevant to the role. His technical skills include proficiency in SQL and Python, as well as experience with data visualization tools like Tableau and Power BI, aligning well with the job's requirements for data analysis and reporting.

Fong's current role as a Pricing Analyst demonstrates his ability to perform data extraction and transformation, develop KPI dashboards, and support data-driven decision-making, which directly correlates with the responsibilities outlined in the job description. His experience in preparing reports and monitoring performance metrics showcases his analytical skills and attention to detail, essential for identifying data quality issues a

In [30]:
# ── Fit Summary: Iteration 1 ──
def fit_summary_v2(jd_text, resume_text):
    prompt = f"""
You are a hiring consultant writing a brief fit assessment.

JOB DESCRIPTION:
{jd_text}

CANDIDATE RESUME:
{resume_text}

Write a 3-4 sentence summary assessing how well this candidate fits the role.
Begin with an overall fit level: Strong, Moderate, or Weak.
Keep your response to 3-4 sentences only.
"""
    return call_llm(prompt)

result_v2 = fit_summary_v2(target_jd, resume_text)
print("=== FIT SUMMARY v2: Goldman Sachs ===\n")
print(result_v2)

=== FIT SUMMARY v2: Goldman Sachs ===

**Fit Level: Strong**  
Fong Lieu possesses a solid educational background in Economics and Business Analytics, complemented by relevant technical skills in SQL, Python, and data visualization tools like Tableau and Power BI. With experience in data extraction, transformation, and KPI dashboard development, he aligns well with the responsibilities of supporting data governance and quality initiatives at Goldman Sachs. His proactive approach to leveraging data for operational improvements and his interest in automation and AI further enhance his suitability for the Analyst role in the XIG Data Office.


In [31]:
# ── Fit Summary: Iteration 2 ──
def fit_summary_v3(jd_text, resume_text):
    prompt = f"""
You are a hiring consultant writing a brief fit assessment.

JOB DESCRIPTION:
{jd_text}

CANDIDATE RESUME:
{resume_text}

Write a 3-4 sentence narrative Fit Summary assessing how well this candidate fits the role.

- Sentence 1: State overall fit level (Strong / Moderate / Weak) and the primary reason.
- Sentence 2: Highlight the candidate's single strongest relevant qualification for this role.
- Sentence 3: Identify the most significant gap between the candidate and the role's requirements.
- Sentence 4 (optional): One specific recommendation to strengthen the application.

Be direct. Keep to 3-4 sentences only.
"""
    return call_llm(prompt)

result_v3 = fit_summary_v3(target_jd, resume_text)
print("=== FIT SUMMARY v3: Goldman Sachs ===\n")
print(result_v3)

=== FIT SUMMARY v3: Goldman Sachs ===

**Fit Summary:** The candidate is a strong fit for the Analyst role in the XIG Data Office at Goldman Sachs due to their solid background in data analysis and experience with relevant tools. Their proficiency in SQL and Python, combined with hands-on experience in developing KPI dashboards, aligns well with the responsibilities of monitoring data quality and supporting data governance initiatives. However, the candidate lacks direct experience in private markets or alternative investments, which is a preferred qualification for the role. To strengthen their application, the candidate should consider highlighting any relevant coursework or projects related to asset management or financial services.


In [32]:
# ── Fit Summary: Iteration 3 ──
def fit_summary(jd_text, resume_text):
    prompt = f"""
You are a hiring consultant writing a brief fit assessment.

JOB DESCRIPTION:
{jd_text}

CANDIDATE RESUME:
{resume_text}

Write a 3-4 sentence narrative Fit Summary assessing how well this candidate fits the role.

- Sentence 1: State overall fit level (Strong / Moderate / Weak) and primary reason,
  citing specific evidence from both documents.
- Sentence 2: Highlight the candidate's single strongest relevant qualification for this role.
- Sentence 3: Identify the most significant gap between the candidate and the role's requirements.
- Sentence 4 (optional): One specific recommendation to strengthen the application.

Be direct and evidence-based. Do not use filler phrases.
Only reference information present in the documents provided.
"""
    return call_llm(prompt)

result_v4 = fit_summary(target_jd, resume_text)
print("=== FIT SUMMARY v4: Goldman Sachs ===\n")
print(result_v4)

=== FIT SUMMARY v4: Goldman Sachs ===

**Fit Summary:** The candidate is a strong fit for the Analyst role in the XIG Data Office at Goldman Sachs due to their solid background in data analysis and experience with relevant tools such as SQL, Python, and Tableau. Their current role as a Pricing Analyst, where they developed KPI dashboards and performed data extraction and transformation, aligns well with the responsibilities of monitoring data quality and supporting data governance initiatives. However, the candidate lacks direct experience in private markets or alternative investments, which is preferred for this position. To strengthen their application, the candidate should emphasize any relevant coursework or projects related to asset management or private markets.


### Prompt Iteration Log

**Iteration 1 — Fit Summary**
Prompt: Vague single-instruction prompt with no structure, length constraint,
or guidance on what to include. Problem: Output was 5-7 sentences, inconsistent
in structure across JDs, never stated a fit level, and used generic language
with no evidence cited.

**Iteration 2 — Fit Summary**
Change: Added explicit fit level requirement (Strong/Moderate/Weak) and
constrained output to 3-4 sentences. Improvement: Outputs became concise and
always led with a clear verdict. Remaining issue: Sentences were still vague —
"has relevant experience" without specifying what experience or which JD
requirement it matched.

**Iteration 3 — Fit Summary**
Change: Added sentence-level instructions specifying what each sentence must
cover: fit verdict, strongest qualification, biggest gap, and an optional
recommendation. Improvement: Output became consistently structured and directly
comparable across all 3 target JDs. Remaining issue: No evidence citations
required, so claims like "demonstrates strong analytical skills" remained
unsupported assertions not tied to either document.

**Iteration 4 — Fit Summary (Final)**
Change: Added evidence citation requirement for every claim and added
faithfulness constraints ("do not use filler phrases," "only reference
information present in the documents provided"). Improvement: All claims now
cite specific resume evidence (e.g., "developed KPI dashboards in Tableau for
a team of 5") matched to specific JD requirements (e.g., "preparing dashboards
for internal stakeholders"). Hallucinated qualifications were eliminated.
This final version was used for all 9 evaluation analyses.

---
<a id="6-comparison"></a>
## 6. Zero-shot vs. Few-shot Comparison

Pick one of your 3 analysis types. Create a few-shot version by adding 1-2 example input/output pairs to the prompt. Run both versions on the same JD and compare outputs.

**Reminder:** You must write the few-shot examples yourself (Tier 2).

In [34]:
# ── Few-shot version of your chosen analysis ──
def skill_gap_report_fewshot(jd_text, resume_text):
    prompt = f"""
You are a career coach analyzing a candidate's resume against a job description.
Below is one example of the correct output format, followed by the actual task.

--- EXAMPLE ---
MATCHING SKILLS:
- SQL: Candidate lists SQL under Technical Skills and used it for data extraction at Onboard Logistics.
- Data Visualization: Candidate built Tableau dashboards for KPI reporting in current role.

MISSING SKILLS:
- dbt: JD requires dbt for pipeline management; not mentioned anywhere in the resume.
- A/B Testing: JD lists experimentation design as required; resume has no mention of this.

RECOMMENDED ACTIONS:
- dbt: Complete the free "dbt Fundamentals" course at courses.getdbt.com and build a personal project.
- A/B Testing: Take a Coursera or DataCamp course on A/B testing and apply it to a Kaggle dataset.
--- END EXAMPLE ---

Now perform the same analysis for the following:

JOB DESCRIPTION:
{jd_text}

CANDIDATE RESUME:
{resume_text}

MATCHING SKILLS:
MISSING SKILLS:
RECOMMENDED ACTIONS:
"""
    return call_llm(prompt)

In [35]:
# ── Run both on the same JD, display side by side ──
comparison_jd = get_jd_text("jd_09_bayview_data_ops_analyst.txt")

zero_shot_output = skill_gap_report(comparison_jd, resume_text)
few_shot_output  = skill_gap_report_fewshot(comparison_jd, resume_text)

print("=" * 60)
print("ZERO-SHOT OUTPUT — Bayview Asset Management")
print("=" * 60)
print(zero_shot_output)

print("\n" + "=" * 60)
print("FEW-SHOT OUTPUT — Bayview Asset Management")
print("=" * 60)
print(few_shot_output)

ZERO-SHOT OUTPUT — Bayview Asset Management
### SKILL GAP REPORT

#### MATCHING SKILLS:
- **SQL**: The candidate lists SQL as a technical skill and has experience with data extraction and transformation in their role as a Pricing Analyst.
- **Python**: Proficient in Python, as evidenced by coursework and projects, including the Gender Pay Inequity Analysis and Airline Sentiment Analysis.
- **Tableau**: Experience developing KPI dashboards in Tableau for a team, which aligns with the requirement for data visualization tools.
- **Data Extraction & Transformation**: Demonstrated experience in data extraction and transformation in the Pricing Analyst role, supporting cost-efficient routing strategies.
- **Excel**: Proficient in Excel, as indicated in the technical skills section.
- **Analytical Skills**: The candidate has performed data analysis and regression modeling in academic projects, showcasing strong analytical capabilities.
- **Communication Skills**: Experience as a Professional 

### Zero-shot vs. Few-shot Analysis

**Which analysis type did you compare?**
Skill Gap Report

**Which performed better?**
Few-shot performed better for output format consistency and conciseness.

**Why? (use specific examples from the outputs above)**
The most visible difference between the two outputs was formatting. The zero-shot
version used markdown styling throughout — bold labels with double asterisks
(e.g., "**ETL Tools**", "**Data Modeling and Data Warehouse Design**") and
section headers with "####" — despite those not being requested. The few-shot
version produced clean plain-text bullets (e.g., "ETL Tools: JD requires
familiarity with ETL tools such as Informatica...") that matched the requested
format exactly. Both versions correctly identified the same four core missing
skills (ETL tools, data warehousing, cloud technologies, financial services
experience), demonstrating that content accuracy was equivalent. However, the
few-shot version was more concise — it listed 4 missing skills compared to
zero-shot's 6, avoiding weaker inferences like "Collateral Management and
Valuation" which had no real basis in the resume. The few-shot example also
produced more specific evidence citations, e.g., "has experience with data
extraction and transformation at Onboard Logistics" vs. zero-shot's vaguer
"experience in data extraction and transformation." Overall, the few-shot
prompt improved output format compliance and reduced low-confidence inferences.

---
<a id="7-evaluation"></a>
## 7. Evaluation

Run all 3 analysis types on your **top 3 target JDs** (9 total analyses).

For each, score:
- **Retrieval relevance:** Did it pull the right JD sections? (Yes/Partial/No)
- **Skill identification accuracy:** Are identified skills/gaps correct? (count correct vs. incorrect)
- **Actionability:** Are recommendations specific and useful? (1-5)
- **Faithfulness:** Does output stick to document content? (Faithful/Partial/Hallucinated)

**Reminder:** Evaluation must be your own work (Tier 2 — AI prohibited).

In [38]:
# ── Run 9 analyses (3 JDs x 3 analysis types) ──
top_3_jds = [
    ("jd_10_goldmansachs_data_office_analyst.txt", "Goldman Sachs"),
    ("jd_09_bayview_data_ops_analyst.txt",          "Bayview Asset Management"),
    ("jd_04_delta_data_analyst.txt",                "Delta Air Lines"),
]

results_store = {}

for filename, company in top_3_jds:
    jd_text = get_jd_text(filename)
    results_store[company] = {
        "skill_gap":         skill_gap_report(jd_text, resume_text),
        "keyword_alignment": keyword_alignment(jd_text, resume_text),
        "fit_summary":       fit_summary(jd_text, resume_text)
    }
    print(f"Completed: {company}")

print("\nAll 9 analyses complete.")

Completed: Goldman Sachs
Completed: Bayview Asset Management
Completed: Delta Air Lines

All 9 analyses complete.


In [39]:
# ── Summarize evaluation results ──
for company, analyses in results_store.items():
    print("=" * 70)
    print(f"COMPANY: {company}")
    print("=" * 70)
    for analysis_type, output in analyses.items():
        print(f"\n--- {analysis_type.upper().replace('_', ' ')} ---")
        print(output)
    print()

COMPANY: Goldman Sachs

--- SKILL GAP ---
### SKILL GAP REPORT

#### MATCHING SKILLS:
- **Data Analysis Experience**: The candidate has experience in data extraction and transformation as a Pricing Analyst at Onboard Logistics, which aligns with the requirement for data analysis and operations experience.
- **Technical Skills**: Proficiency in SQL and Python is demonstrated in the resume, matching the job's requirement for these programming languages.
- **Data Visualization**: The candidate has developed KPI dashboards in Tableau, which supports the need for preparing dashboards and reports for internal stakeholders.
- **Attention to Detail**: The candidate's work in improving win rates and generating gross profit through data-driven strategies indicates strong analytical skills and attention to detail.
- **Project Support**: Experience in tracking actions and timelines as a Professional Learning Facilitator shows capability in supporting project delivery.
- **Stakeholder Engagement**:

In [41]:
# ── Evaluation summary table ──
eval_data = {
    "Company": [
        "Goldman Sachs", "Goldman Sachs",    "Goldman Sachs",
        "Bayview",       "Bayview",          "Bayview",
        "Delta",         "Delta",            "Delta"
    ],
    "Analysis Type": [
        "Skill Gap", "Keyword Alignment", "Fit Summary",
        "Skill Gap", "Keyword Alignment", "Fit Summary",
        "Skill Gap", "Keyword Alignment", "Fit Summary"
    ],
    "Retrieval Relevance": [
        "Yes", "Yes", "Yes",
        "Yes", "Yes", "Yes",
        "Yes", "Yes", "Yes"
    ],
    "Skill ID Accuracy": [
        "5/6 correct", "11/15 correct", "N/A",
        "7/8 correct", "10/15 correct", "N/A",
        "4/5 correct", "11/15 correct", "N/A"
    ],
    "Actionability (1-5)": [
        4, 3, 4,
        4, 3, 4,
        4, 3, 4
    ],
    "Faithfulness": [
        "Partial", "Faithful", "Faithful",
        "Faithful", "Partial", "Faithful",
        "Faithful", "Faithful", "Faithful"
    ]
}

eval_df = pd.DataFrame(eval_data)
display(eval_df)

,Company,Analysis Type,Retrieval Relevance,Skill ID Accuracy,Actionability (1-5),Faithfulness
0,Goldman Sachs,Skill Gap,Yes,5/6 correct,4,Partial
1,Goldman Sachs,Keyword Alignment,Yes,11/15 correct,3,Faithful
2,Goldman Sachs,Fit Summary,Yes,N/A,4,Faithful
3,Bayview,Skill Gap,Yes,7/8 correct,4,Faithful
4,Bayview,Keyword Alignment,Yes,10/15 correct,3,Partial
5,Bayview,Fit Summary,Yes,N/A,4,Faithful
6,Delta,Skill Gap,Yes,4/5 correct,4,Faithful
7,Delta,Keyword Alignment,Yes,11/15 correct,3,Faithful
8,Delta,Fit Summary,Yes,N/A,4,Faithful


### Evaluation Analysis

**Which analysis type worked best?**
The Skill Gap Report produced the most actionable and verifiable results across
all three JDs. Its structured three-section format (Matching Skills, Missing
Skills, Recommended Actions) made it easy to confirm whether each identified
skill was actually present in the resume and whether each gap was genuinely
absent. The recommended actions were consistently specific — naming actual
certifications (DAMA Certified Data Management Professional), platforms
(Coursera, AWS), and tools (Apache Airflow, Talend) — earning an actionability
score of 4 across all three companies.

**Which JDs produced the best/worst results? Why?**
Bayview Asset Management produced the strongest results overall. The JD is
highly technical with clearly defined requirements (specific ETL tools by name,
data warehouse design, cloud technologies), giving the model concrete, verifiable
items to compare against the resume. This produced a precise skill gap with
minimal ambiguity. Delta Air Lines also performed well, particularly in keyword
alignment, where the model correctly identified partial matches like "Onboard
Logistics" as logistics experience relevant to the aviation/transportation
preference. Goldman Sachs produced the weakest results due to over-inference —
the Skill Gap report listed "Stakeholder Engagement" as a match based solely on
the candidate's teaching role, which is a loose connection to a data governance
context. The JD's softer preferred qualifications (interest in private markets,
curiosity about AI) were harder for the model to evaluate objectively.

**Where did the system hallucinate or produce inaccurate results?**
The most notable faithfulness issue occurred in the Bayview Keyword Alignment
analysis, which marked "Communication skills — MISSING" despite the resume
explicitly listing communication skills under Technical Skills and the candidate's
entire teaching role being communication-focused. This appears to be a case where
the model searched for the exact phrase "communication skills" as a keyword rather
than recognizing it semantically. A similar issue appeared in Goldman Sachs Skill
Gap, where "Project Support" was listed as a match based on the teaching role's
"tracking actions and timelines" — a stretch that conflates classroom facilitation
with data project management.

**What would you improve?**
The most impactful improvement would be filtering vector store retrieval by JD
filename so only the target document's chunks are used in each analysis, rather
than potentially pulling from other JDs in the corpus. This would eliminate any
cross-contamination between JDs during retrieval. Additionally, adding a
post-processing step to verify that each identified matching skill contains an
explicit quote from the resume would reduce loose inferences like the stakeholder
engagement example. Finally, the Keyword Alignment prompt could be improved by
instructing the model to check for semantic equivalence more carefully before
marking a skill as missing, which would have prevented the "communication skills"
misclassification in the Bayview analysis.

---

## Next Steps

1. Build your Streamlit app (`streamlit_app.py`) using the pipeline from this notebook
2. Write your Technical Manager Memo (`memo.md`)
3. Complete your AI Usage Log (`ai_log.md`)
4. Verify GitHub repository structure and commit count

---
*BSAN 6200 | Spring 2026 | Assignment 5 — Option B*